# Reference: MLflow Judges & Scorers

---

This notebook is a Python API reference for MLflow's evaluation scorers — the functions that assess LLM output quality.

**What this covers:**

| Category | What | Section |
|---|---|---|
| Built-in LLM judges | `Correctness`, `Safety`, `RelevanceToQuery`, etc. | §1 |
| Guidelines judges | `Guidelines`, `ExpectationsGuidelines` | §2 |
| RAG judges | `RetrievalGroundedness`, `RetrievalSufficiency`, `RetrievalRelevance` | §3 |
| Tool call judges | `ToolCallCorrectness`, `ToolCallEfficiency` | §4 |
| Custom code-based scorers | `@scorer` decorator, `Scorer` class | §5 |
| Custom LLM judges | `make_judge()` | §6 |
| Scorer versioning | `register()`, `get_scorer()`, `list_scorers()` | §7 |

> **Requirements:** MLflow 3.2+. Built-in LLM judges use OpenAI by default — set `OPENAI_API_KEY` or pass a custom `model` parameter.

---
## 0 — Setup

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("scorers-reference")

print(f"MLflow version: {mlflow.__version__}")

---
## 1 — Built-in LLM Judges

MLflow ships pre-configured judges that evaluate common quality dimensions. Each judge reads the trace and/or dataset fields and returns pass/fail feedback with a rationale.

### Full list of built-in judges

| Category | Judges |
|---|---|
| **Response quality** | `Correctness`, `RelevanceToQuery`, `Safety`, `Completeness`, `Fluency`, `Equivalence`, `Summarization` |
| **Guidelines** | `Guidelines`, `ExpectationsGuidelines` |
| **RAG** | `RetrievalGroundedness`, `RetrievalSufficiency`, `RetrievalRelevance` |
| **Tool calls** | `ToolCallCorrectness`, `ToolCallEfficiency` |
| **Multi-turn** | `ConversationCompleteness`, `ConversationalGuidelines`, `ConversationalRoleAdherence`, `ConversationalSafety`, `ConversationalToolCallEfficiency`, `KnowledgeRetention`, `UserFrustration` |

All built-in judges are importable from `mlflow.genai.scorers`.

### Response quality judges

| Judge | What it checks | Requires |
|---|---|---|
| `Correctness()` | Factual accuracy against expected facts | `expectations.expected_facts` |
| `RelevanceToQuery()` | Does the response address the question? | `inputs` only |
| `Safety()` | Checks for harmful or unethical content | `outputs` only |
| `Completeness()` | Does the response cover all aspects? | `expectations.expected_facts` |
| `Fluency()` | Grammar, readability, coherence | `outputs` only |
| `Equivalence()` | Is the response equivalent to a reference? | `expectations.expected_response` |
| `Summarization()` | Quality of summarization | `inputs` (source text) |

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety

# Minimal dataset — pre-computed outputs (no predict_fn needed)
data = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "outputs": "The capital of France is Paris.",
        "expectations": {
            "expected_facts": ["The capital of France is Paris."],
        },
    },
    {
        "inputs": {"question": "What is the boiling point of water?"},
        "outputs": "Water boils at 100 degrees Celsius at sea level.",
        "expectations": {
            "expected_facts": ["100 degrees Celsius", "212 degrees Fahrenheit"],
        },
    },
]

results = mlflow.genai.evaluate(
    data=data,
    scorers=[Correctness(), RelevanceToQuery(), Safety()],
)

results.tables["eval_results_table"]

---
## 2 — Guidelines Judges

Guidelines judges evaluate responses against natural-language criteria. Two variants:

| Judge | Guidelines source | Use case |
|---|---|---|
| `Guidelines(name, guidelines)` | Same criteria for **all** rows | Brand voice, formatting standards |
| `ExpectationsGuidelines()` | Per-row via `expectations.guidelines` | Row-specific acceptance criteria |

### Global guidelines (same for every row)

In [ ]:
from mlflow.genai.scorers import Guidelines

# Single guideline as a string
conciseness = Guidelines(
    name="conciseness",
    guidelines="The response must be under 3 sentences.",
)

# Multiple guidelines as a list — ALL must pass
tone_check = Guidelines(
    name="professional_tone",
    guidelines=[
        "The response must not use slang or informal language.",
        "The response must not include emojis.",
        "The response must address the user respectfully.",
    ],
)

data = [
    {
        "inputs": {"question": "How do I reset my password?"},
        "outputs": "Go to Settings > Security > Reset Password. Follow the prompts.",
    },
    {
        "inputs": {"question": "Why is my order late?"},
        "outputs": "lol idk maybe check the tracking link? 🤷",
    },
]

results = mlflow.genai.evaluate(
    data=data,
    scorers=[conciseness, tone_check],
)

results.tables["eval_results_table"]

### Per-row guidelines (different criteria per example)

In [ ]:
from mlflow.genai.scorers import ExpectationsGuidelines

data = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "outputs": "The capital of France is Paris.",
        "expectations": {
            "guidelines": ["The response must be factual and concise."],
        },
    },
    {
        "inputs": {"question": "How to learn Python?"},
        "outputs": "You can read a book or take a course.",
        "expectations": {
            "guidelines": [
                "The response must be helpful and encouraging.",
                "The response must suggest at least two specific resources.",
            ],
        },
    },
]

results = mlflow.genai.evaluate(
    data=data,
    scorers=[ExpectationsGuidelines()],
)

results.tables["eval_results_table"]

### Using a custom model for guidelines evaluation

In [ ]:
# Override the default judge model
strict_check = Guidelines(
    name="strict_formatting",
    guidelines="The response must use bullet points for any list of items.",
    model="openai:/gpt-4o-mini",  # specify the judge model
)

---
## 3 — RAG Judges

These judges evaluate retrieval-augmented generation pipelines by inspecting both the retrieved context and the final response.

| Judge | What it checks | Reads from |
|---|---|---|
| `RetrievalGroundedness()` | Is the response supported by retrieved context? | Trace (RETRIEVER spans) |
| `RetrievalSufficiency()` | Was the retrieved context sufficient to answer? | Trace (RETRIEVER spans) + `expectations` |
| `RetrievalRelevance()` | Are the retrieved documents relevant to the query? | Trace (RETRIEVER spans) |

RAG judges read **RETRIEVER** spans from the trace automatically. Your retrieval function must be traced with `span_type=SpanType.RETRIEVER`.

In [ ]:
from mlflow.genai.scorers import RetrievalGroundedness, RetrievalSufficiency, RetrievalRelevance
from mlflow.entities import SpanType

# Example: a simple RAG pipeline with traced retrieval
@mlflow.trace
def rag_pipeline(question: str) -> str:
    chunks = retrieve(question)
    context = "\n".join(chunks)
    return f"Based on the context: {context}. The answer is: example response."


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve(query: str) -> list[str]:
    # Simulated retrieval — in practice, this queries a vector DB
    return [
        "Paris is the capital of France.",
        "France is a country in Western Europe.",
    ]


# RAG judges work with predict_fn — they inspect the trace
data = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_facts": ["Paris"]},
    },
]

results = mlflow.genai.evaluate(
    data=data,
    predict_fn=rag_pipeline,
    scorers=[RetrievalGroundedness(), RetrievalSufficiency(), RetrievalRelevance()],
)

results.tables["eval_results_table"]

---
## 4 — Tool Call Judges

These judges evaluate whether an agent called the right tools with the right arguments.

| Judge | What it checks | Requires |
|---|---|---|
| `ToolCallCorrectness()` | Right tool, right arguments (fuzzy match) | `expectations.expected_tool_calls` |
| `ToolCallEfficiency()` | No redundant or duplicate calls | Trace only (no ground truth) |

Both judges read **TOOL** spans from the trace. The Agents SDK and `@mlflow.trace(span_type=SpanType.TOOL)` both create these spans.

In [ ]:
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency

# Tool call judges require a predict_fn that generates traces with TOOL spans.
# The expected_tool_calls field defines the ground truth.

tool_eval_data = [
    {
        "inputs": {"question": "How many grams is 2 cups of flour?"},
        "expectations": {
            "expected_tool_calls": [
                {
                    "name": "convert_units",
                    "arguments": {
                        "amount": 2,
                        "from_unit": "cups",
                        "to_unit": "grams",
                        "ingredient": "flour",
                    },
                },
            ],
        },
    },
    {
        # No tool should be called for general questions
        "inputs": {"question": "What's the best way to chop an onion?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
]

# Note: this requires a predict_fn that actually calls tools.
# results = mlflow.genai.evaluate(
#     data=tool_eval_data,
#     predict_fn=my_agent_predict_fn,
#     scorers=[ToolCallCorrectness(), ToolCallEfficiency()],
# )

### `expected_tool_calls` format

```python
"expected_tool_calls": [
    {
        "name": "tool_name",           # required — which tool
        "arguments": {                  # optional — expected arguments
            "param1": "value1",
            "param2": 42,
        },
    },
]
```

- `ToolCallCorrectness` uses **fuzzy matching** — argument values don't need to be exact.
- An empty list `[]` means the agent should NOT call any tools.
- Multiple entries mean the agent should call multiple tools (order-independent).

---
## 5 — Custom Code-Based Scorers

When built-in judges don't fit your needs, write your own with the `@scorer` decorator. Custom scorers are plain Python functions — no LLM calls required.

### Function signature

```python
from mlflow.genai.scorers import scorer

@scorer
def my_scorer(
    *,                                          # keyword-only
    inputs: dict[str, Any] | None,              # the raw input
    outputs: Any | None,                        # the raw output
    expectations: dict[str, Any] | None,        # ground truth
    trace: mlflow.entities.Trace | None,        # the full trace
) -> int | float | bool | str | Feedback | list[Feedback]:
    ...
```

You only need to include the parameters you use — MLflow inspects the signature.

### Return types

| Return type | Display | Example |
|---|---|---|
| `bool` | True/False | `return len(outputs) > 0` |
| `"yes"` / `"no"` | Pass/Fail | `return "yes" if valid else "no"` |
| `int` / `float` | Numeric score | `return len(outputs.split())` |
| `Feedback` | Value + rationale | Detailed assessment |
| `list[Feedback]` | Multiple metrics | Multi-aspect scoring |

### Simple return values

In [ ]:
from mlflow.genai.scorers import scorer


@scorer
def word_count(outputs: str) -> int:
    """Count words in the response."""
    return len(outputs.split())


@scorer
def is_concise(outputs: str) -> bool:
    """Pass if the response is under 50 words."""
    return len(outputs.split()) < 50


@scorer
def has_citation(outputs: str) -> str:
    """Check if output contains a citation."""
    return "yes" if "[source]" in outputs.lower() or "http" in outputs.lower() else "no"


data = [
    {"inputs": {"q": "What is Python?"}, "outputs": "Python is a programming language."},
    {"inputs": {"q": "What is Rust?"}, "outputs": "Rust is a systems programming language focused on safety. See [source] https://rust-lang.org"},
]

results = mlflow.genai.evaluate(data=data, scorers=[word_count, is_concise, has_citation])
results.tables["eval_results_table"]

### Returning `Feedback` objects

Use `Feedback` when you need to include a rationale or metadata alongside the score.

In [ ]:
from mlflow.entities import Feedback


@scorer
def length_check(outputs: str) -> Feedback:
    """Check if response length is within acceptable range."""
    word_count = len(outputs.split())
    is_ok = 10 <= word_count <= 200
    return Feedback(
        value=is_ok,
        rationale=f"Response has {word_count} words. {'Within' if is_ok else 'Outside'} acceptable range (10-200).",
    )

### Returning multiple `Feedback` objects

A single scorer can produce multiple named metrics by returning a list of `Feedback` objects.

In [ ]:
@scorer
def multi_check(inputs: dict, outputs: str) -> list[Feedback]:
    """Evaluate multiple dimensions in one scorer."""
    words = outputs.split()
    return [
        Feedback(name="word_count", value=len(words), rationale=f"{len(words)} words"),
        Feedback(name="mentions_input", value="yes" if any(v.lower() in outputs.lower() for v in inputs.values()) else "no"),
        Feedback(name="has_punctuation", value=outputs.rstrip()[-1] in ".!?" if outputs else False),
    ]

### Using `expectations` (ground truth)

In [ ]:
@scorer
def exact_match(outputs: str, expectations: dict) -> bool:
    """Check if the output exactly matches the expected response."""
    return outputs.strip() == expectations["expected_response"].strip()


@scorer
def contains_all_facts(outputs: str, expectations: dict) -> Feedback:
    """Check if all expected facts appear in the output."""
    facts = expectations.get("expected_facts", [])
    found = [f for f in facts if f.lower() in outputs.lower()]
    missing = [f for f in facts if f.lower() not in outputs.lower()]
    return Feedback(
        value=len(found) / len(facts) if facts else 1.0,
        rationale=f"Found {len(found)}/{len(facts)} facts. Missing: {missing}" if missing else "All facts present.",
    )


data = [
    {
        "inputs": {"q": "Capital of France?"},
        "outputs": "Paris",
        "expectations": {"expected_response": "Paris", "expected_facts": ["Paris"]},
    },
    {
        "inputs": {"q": "First 3 planets?"},
        "outputs": "Mercury and Venus.",
        "expectations": {"expected_response": "Mercury, Venus, Earth", "expected_facts": ["Mercury", "Venus", "Earth"]},
    },
]

results = mlflow.genai.evaluate(data=data, scorers=[exact_match, contains_all_facts])
results.tables["eval_results_table"]

### Using `trace` for span-level inspection

Scorers can inspect the full execution trace — useful for checking tool calls, retrieval spans, or latency.

In [ ]:
from mlflow.entities import SpanType, Trace


@scorer
def tool_call_trajectory(trace: Trace, expectations: dict) -> Feedback:
    """Check if the agent called tools in the expected order."""
    tool_spans = trace.search_spans(span_type=SpanType.TOOL)
    actual = [span.name for span in tool_spans]
    expected = expectations["tool_call_trajectory"]

    if actual == expected:
        return Feedback(value=1, rationale="Tool call trajectory is correct.")
    return Feedback(
        value=0,
        rationale=f"Expected: {expected}. Actual: {actual}.",
    )


@scorer
def retrieval_recall(trace: Trace, expectations: dict) -> Feedback:
    """Check what fraction of expected documents were retrieved."""
    retriever_spans = trace.search_spans(span_type=SpanType.RETRIEVER)
    if not retriever_spans:
        return Feedback(value=0, rationale="No retriever span found.")

    retrieved_urls = []
    for span in retriever_spans:
        retrieved_urls.extend([doc["doc_uri"] for doc in span.outputs])

    expected_urls = expectations["relevant_document_urls"]
    hits = len(set(retrieved_urls) & set(expected_urls))
    recall = hits / len(expected_urls) if expected_urls else 1.0
    return Feedback(
        value=recall,
        rationale=f"Retrieved {hits}/{len(expected_urls)} relevant documents.",
    )


@scorer
def is_routing_correct(trace: Trace, expectations: dict) -> Feedback:
    """Check if the right sub-agents were invoked."""
    agent_spans = trace.search_spans(span_type=SpanType.AGENT)
    invoked = [span.name for span in agent_spans]
    expected = expectations["expected_agents"]

    if invoked == expected:
        return Feedback(value=True, rationale="Sub-agent routing is correct.")
    return Feedback(
        value=False,
        rationale=f"Expected: {expected}. Actual: {invoked}.",
    )

### `Scorer` class (for configurable scorers)

When you need a scorer with configuration parameters, use the `Scorer` base class instead of the decorator.

In [ ]:
from mlflow.genai.scorers import Scorer


class WordCountRange(Scorer):
    """Check if word count falls within a configurable range."""
    name: str = "word_count_range"
    min_words: int = 10
    max_words: int = 200

    def __call__(self, outputs: str) -> Feedback:
        count = len(outputs.split())
        in_range = self.min_words <= count <= self.max_words
        return Feedback(
            value=in_range,
            rationale=f"{count} words. Range: [{self.min_words}, {self.max_words}].",
        )


# Use with different configurations
short_check = WordCountRange(name="short_check", min_words=1, max_words=50)
long_check = WordCountRange(name="long_check", min_words=50, max_words=500)

### Error handling in scorers

Scorers can either let exceptions propagate (MLflow captures them) or return explicit errors.

In [ ]:
import json
from mlflow.entities import AssessmentError


@scorer
def validate_json_output(outputs: str) -> Feedback:
    """Check if the output is valid JSON with required fields."""
    try:
        data = json.loads(outputs)
        required = ["summary", "confidence"]
        missing = [f for f in required if f not in data]
        if missing:
            return Feedback(
                error=AssessmentError(
                    error_code="MISSING_FIELDS",
                    error_message=f"Missing required fields: {missing}",
                ),
            )
        return Feedback(value=True, rationale="Valid JSON with all required fields.")
    except json.JSONDecodeError as e:
        return Feedback(error=e)

---
## 6 — Custom LLM Judges with `make_judge()`

When you need an LLM to evaluate quality but the built-in judges don't fit, create a custom judge with `make_judge()`. You write natural-language instructions, and MLflow handles the LLM call, parsing, and feedback logging.

### Template variables

| Variable | Resolves to |
|---|---|
| `{{ inputs }}` | The user's input |
| `{{ outputs }}` | The model's response |
| `{{ expectations }}` | Ground truth / expected behavior |
| `{{ trace }}` | Full execution trace (makes this a trace-based judge) |

### Basic judge (inputs + outputs)

In [ ]:
from typing import Literal
from mlflow.genai.judges import make_judge

# Categorical feedback
helpfulness_judge = make_judge(
    name="helpfulness",
    instructions=(
        "Evaluate if the response in {{ outputs }} is helpful for the user's question in {{ inputs }}.\n\n"
        "A helpful response directly addresses the question, provides actionable information, "
        "and doesn't include unnecessary filler."
    ),
    feedback_value_type=Literal["very_helpful", "somewhat_helpful", "not_helpful"],
)

### Boolean judge

In [ ]:
# Boolean (yes/no) feedback
politeness_judge = make_judge(
    name="politeness",
    instructions=(
        "Determine if the response in {{ outputs }} maintains a polite and professional tone.\n\n"
        "The response should not be dismissive, rude, or use inappropriate language."
    ),
    feedback_value_type=bool,
)

### Numeric score judge

In [ ]:
# Integer score (e.g. 1-100)
reasoning_judge = make_judge(
    name="reasoning_quality",
    instructions=(
        "Evaluate the reasoning quality in {{ outputs }} for the question in {{ inputs }}.\n\n"
        "Score 1-100 based on:\n"
        "- Logical progression (does each step follow from the previous?)\n"
        "- Evidence usage (are claims supported?)\n"
        "- Conclusion soundness (does the conclusion follow from the premises?)"
    ),
    feedback_value_type=int,
)

### Judge with expectations

In [ ]:
# Compare output against expected behaviors
behavior_judge = make_judge(
    name="expected_behaviors",
    instructions=(
        "Compare the agent's response in {{ outputs }} against the expected behaviors in {{ expectations }}.\n\n"
        "User's question: {{ inputs }}\n\n"
        "Determine if the response meets, partially meets, or does not meet the expected behaviors."
    ),
    feedback_value_type=Literal["meets_expectations", "partially_meets", "does_not_meet"],
)

### Trace-based judge

Including `{{ trace }}` in the instructions makes the judge trace-aware — it can inspect tool calls, retrieval spans, latency, etc.

In [ ]:
# Trace-based — inspects the full execution
tool_usage_judge = make_judge(
    name="tool_usage_quality",
    instructions=(
        "Analyze the execution {{ trace }} to evaluate tool usage.\n\n"
        "Check for:\n"
        "1. Were the right tools called for this request?\n"
        "2. Were tool arguments correct and complete?\n"
        "3. Were there any unnecessary or redundant tool calls?\n"
        "4. Was the tool output used properly in the final response?"
    ),
    feedback_value_type=Literal["optimal", "good", "suboptimal", "poor"],
    model="openai:/gpt-4o-mini",  # model is required for trace-based judges
)

### Using custom judges in evaluation

In [ ]:
data = [
    {
        "inputs": {"question": "How do I reset my password?"},
        "outputs": "Go to Settings > Security > Reset Password and follow the prompts.",
        "expectations": {
            "expected_behaviors": ["Provide step-by-step instructions", "Mention the Settings menu"],
        },
    },
    {
        "inputs": {"question": "Why is my order late?"},
        "outputs": "idk check your email lol",
        "expectations": {
            "expected_behaviors": ["Apologize for the delay", "Offer to check order status"],
        },
    },
]

results = mlflow.genai.evaluate(
    data=data,
    scorers=[helpfulness_judge, politeness_judge, behavior_judge],
)

results.tables["eval_results_table"]

---
## 7 — Scorer Versioning & Registration

Scorers (including custom judges) can be registered to an MLflow experiment for version tracking and retrieval.

In [ ]:
# Register a scorer to the active experiment
registered = helpfulness_judge.register()
print(f"Registered: {registered}")

In [ ]:
# Update — register a new version with the same name
helpfulness_v2 = make_judge(
    name="helpfulness",
    instructions=(
        "Evaluate if {{ outputs }} is helpful, accurate, and complete for {{ inputs }}.\n\n"
        "Consider: Does it directly answer the question? Is the information actionable? "
        "Does it avoid unnecessary filler or hedging?"
    ),
    feedback_value_type=Literal["very_helpful", "somewhat_helpful", "not_helpful"],
)

registered_v2 = helpfulness_v2.register()
print(f"Registered v2: {registered_v2}")

In [ ]:
from mlflow.genai import get_scorer, list_scorers

# Load the latest version of a registered scorer
latest = get_scorer(name="helpfulness")
print(f"Loaded scorer: {latest}")

# List all registered scorers in the active experiment
all_scorers = list_scorers()
for s in all_scorers:
    print(f"  {s.name}")

---
## Quick Reference

### Imports

```python
# Built-in judges
from mlflow.genai.scorers import (
    Correctness, RelevanceToQuery, Safety, Completeness, Fluency,
    Guidelines, ExpectationsGuidelines,
    RetrievalGroundedness, RetrievalSufficiency, RetrievalRelevance,
    ToolCallCorrectness, ToolCallEfficiency,
)

# Custom scorers
from mlflow.genai.scorers import scorer, Scorer
from mlflow.entities import Feedback, AssessmentError

# Custom LLM judges
from mlflow.genai.judges import make_judge

# Versioning
from mlflow.genai import get_scorer, list_scorers
```

### Decision guide

| Need | Use |
|---|---|
| Standard quality check (correctness, safety, relevance) | Built-in judge |
| Custom pass/fail rules in natural language | `Guidelines(name, guidelines)` |
| Different rules per eval row | `ExpectationsGuidelines()` |
| Deterministic check (length, format, regex) | `@scorer` decorator |
| Configurable deterministic check | `Scorer` class |
| Custom LLM evaluation with specific criteria | `make_judge()` |
| Inspect trace spans (tools, retrieval, agents) | `@scorer` with `trace` parameter |
| Track scorer versions across experiments | `.register()` + `get_scorer()` |

### Dataset field requirements

| Field | Used by |
|---|---|
| `inputs` | All judges (the user's question) |
| `outputs` | All judges (the model's response) |
| `expectations.expected_facts` | `Correctness`, `Completeness` |
| `expectations.expected_response` | `Equivalence` |
| `expectations.guidelines` | `ExpectationsGuidelines` |
| `expectations.expected_tool_calls` | `ToolCallCorrectness` |
| Trace (automatic) | RAG judges, `ToolCallEfficiency`, trace-based `make_judge()` |